# 02 — Data Cleaning & Quality Checks
تنظيف قابل للتتبع مع فحص المفاتيح والعلاقات وبناء dataset تحليلي موحّد.

In [1]:
from pathlib import Path
import pandas as pd

RAW, OUT = Path('../data/raw'), Path('../data/processed')
OUT.mkdir(parents=True, exist_ok=True)
tables = {p.stem: pd.read_csv(p) for p in RAW.glob('*.csv')}
cleaning_log, quality_report = [], []
print('Tables loaded:', list(tables) or 'None')

Tables loaded: ['accounts', 'branches', 'cards', 'customers', 'loans', 'merchants', 'products', 'subscriptions', 'transactions']


In [2]:
def clean_frame(name, frame):
    result = frame.copy()
    before = len(result)
    duplicate_count = int(result.duplicated().sum())
    result = result.drop_duplicates().reset_index(drop=True)
    cleaning_log.append({'table': name, 'rows_before': before, 'rows_after': len(result), 'exact_duplicates_removed': duplicate_count})
    return result

cleaned = {name: clean_frame(name, frame) for name, frame in tables.items()}
pd.DataFrame(cleaning_log)

,table,rows_before,rows_after,exact_duplicates_removed
0,accounts,600,600,0
1,branches,12,12,0
2,cards,358,358,0
3,customers,200,200,0
4,loans,95,95,0
5,merchants,254,254,0
6,products,24,24,0
7,subscriptions,1333,1333,0
8,transactions,476202,476202,0


In [3]:
def check_key(table, key):
    if table not in cleaned or key not in cleaned[table].columns:
        return {'check': f'{table}.{key}', 'status': 'not_checked', 'detail': 'table or column not found'}
    series = cleaned[table][key]
    return {'check': f'{table}.{key}', 'status': 'pass' if series.notna().all() and series.is_unique else 'review', 'detail': f'nulls={series.isna().sum()}, duplicate_values={series.duplicated().sum()}'}

def check_relation(child_table, child_key, parent_table, parent_key):
    if any(t not in cleaned for t in [child_table, parent_table]) or any(k not in cleaned[t].columns for t, k in [(child_table, child_key), (parent_table, parent_key)]):
        return {'check': f'{child_table}.{child_key} -> {parent_table}.{parent_key}', 'status': 'not_checked', 'detail': 'table or column not found'}
    missing = set(cleaned[child_table][child_key].dropna()) - set(cleaned[parent_table][parent_key].dropna())
    return {'check': f'{child_table}.{child_key} -> {parent_table}.{parent_key}', 'status': 'pass' if not missing else 'review', 'detail': f'unmatched_values={len(missing)}'}

for table, key in [('customers','customer_id'), ('accounts','account_id'), ('transactions','transaction_id')]: quality_report.append(check_key(table, key))
for relation in [('accounts','customer_id','customers','customer_id'), ('transactions','account_id','accounts','account_id')]: quality_report.append(check_relation(*relation))
quality_df = pd.DataFrame(quality_report)
quality_df

,check,status,detail
0,customers.customer_id,pass,"nulls=0, duplicate_values=0"
1,accounts.account_id,pass,"nulls=0, duplicate_values=0"
2,transactions.transaction_id,pass,"nulls=0, duplicate_values=0"
3,accounts.customer_id -> customers.customer_id,pass,unmatched_values=0
4,transactions.account_id -> accounts.account_id,pass,unmatched_values=0


### Currency normalization
The synthetic source mixes currencies. All value KPIs use `amount_usd` (illustrative fixed FX rates, not market data).

In [4]:
# Category normalization: the generator emits some categories in French next to the English ones.
# Map them to one English vocabulary so category totals are not split in two; keep the original in category_source.
CATEGORY_MAP = {
    'alimentation': 'groceries', 'restauration': 'dining', 'voyages': 'travel', 'sante': 'health',
    'abonnements': 'subscriptions', 'revenus': 'income', 'logement': 'housing', 'energie': 'utilities',
    'professionnel': 'business', 'famille': 'family',
}
if 'transactions' in cleaned and 'category' in cleaned['transactions'].columns:
    t = cleaned['transactions']
    t['category_source'] = t['category']
    t['category'] = t['category'].astype(str).str.strip().str.lower().replace(CATEGORY_MAP)

# Multi-currency handling: the source mixes USD / GBP / EUR. Amounts are never summed in native currency.
# Convert to a USD reporting currency with documented ILLUSTRATIVE fixed rates (data/reference/fx_rates.csv).
fx = pd.read_csv('../data/reference/fx_rates.csv')
rate = dict(zip(fx['currency'], fx['usd_per_unit']))
if 'transactions' in cleaned and 'currency' in cleaned['transactions'].columns:
    t = cleaned['transactions']
    unknown = sorted(set(t['currency'].dropna()) - set(rate))
    t['usd_per_unit'] = t['currency'].map(rate)
    t['amount_usd'] = (t['amount'] * t['usd_per_unit']).round(2)
    if 'signed_amount' in t.columns:
        t['signed_amount_usd'] = (t['signed_amount'] * t['usd_per_unit']).round(2)
    quality_report.append({'check': 'transactions.currency -> fx_rates', 'status': 'pass' if not unknown and t['amount_usd'].notna().all() else 'review', 'detail': f'currencies={sorted(t["currency"].dropna().unique())}, unconverted_rows={int(t["amount_usd"].isna().sum())}'})
    quality_df = pd.DataFrame(quality_report)
quality_df


,check,status,detail
0,customers.customer_id,pass,"nulls=0, duplicate_values=0"
1,accounts.account_id,pass,"nulls=0, duplicate_values=0"
2,transactions.transaction_id,pass,"nulls=0, duplicate_values=0"
3,accounts.customer_id -> customers.customer_id,pass,unmatched_values=0
4,transactions.account_id -> accounts.account_id,pass,unmatched_values=0
5,transactions.currency -> fx_rates,pass,"currencies=['EUR', 'GBP', 'USD'], unconverted_..."


In [5]:
# Build a unified analytical dataset when the expected relationships exist.
final_dataset = None
if all(t in cleaned for t in ['transactions', 'accounts']):
    tx = cleaned['transactions'].merge(cleaned['accounts'], on='account_id', how='left', suffixes=('', '_account'))
    if 'customers' in cleaned and 'customer_id' in tx.columns:
        tx = tx.merge(cleaned['customers'], on='customer_id', how='left', suffixes=('', '_customer'))
    if 'branches' in cleaned and 'branch_id' in tx.columns:
        tx = tx.merge(cleaned['branches'], on='branch_id', how='left', suffixes=('', '_branch'))
    if 'products' in cleaned and 'product_id' in tx.columns:
        tx = tx.merge(cleaned['products'], on='product_id', how='left', suffixes=('', '_product'))
    final_dataset = tx
    final_dataset.to_csv(OUT / 'final_dataset.csv', index=False)
    print('final_dataset.csv:', final_dataset.shape)
else:
    print('final_dataset.csv not created: transactions and accounts are required.')

final_dataset.csv: (476202, 73)


In [6]:
for name, frame in cleaned.items(): frame.to_csv(OUT / f'cleaned_{name}.csv', index=False)
pd.DataFrame(cleaning_log).to_csv(OUT / 'cleaning_log.csv', index=False)
quality_df.to_csv(OUT / 'quality_report.csv', index=False)
print(f'Saved cleaned tables and quality reports to {OUT.resolve()}')

Saved cleaned tables and quality reports to C:\Users\user\OneDrive\Desktop\Financial Customer & Transaction Analytics\data\processed


## Rules to finalize after schema review
- Parse dates only after confirming date columns and timezone assumptions.
- Standardize categories using an explicit mapping table.
- Handle missing values according to field meaning, not one global rule.
- Review unmatched foreign keys before trusting the final dataset.
- Keep rejected rows and reasons in a separate audit output.